<a href="https://colab.research.google.com/github/Arsh-21/Robustness-Bias-Analysis-of-NLI-Models-SNLI-/blob/main/Robustness_%26_Bias_Analysis_of_NLI_Models_(SNLI).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers datasets evaluate scikit-learn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.8 MB/s eta 0:00:00


In [2]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)
import evaluate


In [3]:
dataset = load_dataset("snli")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/412k [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/413k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/19.6M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/550152 [00:00<?, ? examples/s]

In [22]:
dataset


DatasetDict({
    test: Dataset({
        features: ['premise', 'hypothesis', 'label'],
        num_rows: 9824
    })
    validation: Dataset({
        features: ['premise', 'hypothesis', 'label'],
        num_rows: 9842
    })
    train: Dataset({
        features: ['premise', 'hypothesis', 'label'],
        num_rows: 549367
    })
})

In [28]:
dataset['train'][0:5]

{'premise': ['A person on a horse jumps over a broken down airplane.',
  'A person on a horse jumps over a broken down airplane.',
  'A person on a horse jumps over a broken down airplane.',
  'Children smiling and waving at camera',
  'Children smiling and waving at camera'],
 'hypothesis': ['A person is training his horse for a competition.',
  'A person is at a diner, ordering an omelette.',
  'A person is outdoors, on a horse.',
  'They are smiling at their parents',
  'There are children present'],
 'label': [1, 2, 0, 1, 0]}

In [5]:
def filter_invalid(example):
    return example["label"] != -1

dataset = dataset.filter(filter_invalid)



Filter:   0%|          | 0/10000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/550152 [00:00<?, ? examples/s]

In [6]:
small_train = dataset["train"].shuffle(seed=42).select(range(5000))
small_val = dataset["validation"].shuffle(seed=42).select(range(1000))



In [7]:
def preprocess(example):
    example["combined"] = example["hypothesis"] + " [SEP] " + example["premise"]
    return example

small_train = small_train.map(preprocess)
small_val = small_val.map(preprocess)



Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [8]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize(example):
    return tokenizer(
        example["combined"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

small_train = small_train.map(tokenize, batched=True)
small_val = small_val.map(tokenize, batched=True)



tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [9]:
small_train = small_train.rename_column("label", "labels")
small_val = small_val.rename_column("label", "labels")

small_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
small_val.set_format("torch", columns=["input_ids", "attention_mask", "labels"])


In [10]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)


In [12]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True
)


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_val,
    compute_metrics=compute_metrics
)


In [14]:
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.651305,0.726000
2,0.800391,0.600149,0.764000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=626, training_loss=0.7601318115624376, metrics={'train_runtime': 226.18, 'train_samples_per_second': 44.213, 'train_steps_per_second': 2.768, 'total_flos': 662348805120000.0, 'train_loss': 0.7601318115624376, 'epoch': 2.0})

In [15]:
results = trainer.evaluate()
print(results)


{'eval_loss': 0.6001486778259277, 'eval_accuracy': 0.764, 'eval_runtime': 6.5605, 'eval_samples_per_second': 152.428, 'eval_steps_per_second': 9.603, 'epoch': 2.0}


In [16]:
import random
from datasets import Dataset

def swap_evidence(dataset):
    evidences = []

    for ex in dataset:
        parts = ex["combined"].split("[SEP]")
        if len(parts) > 1:
            evidences.append(parts[1].strip())
        else:
            evidences.append("")

    swapped_examples = []

    for ex in dataset:
        claim = ex["combined"].split("[SEP]")[0].strip()
        random_evidence = random.choice(evidences)

        new_text = claim + " [SEP] " + random_evidence

        new_ex = ex.copy()
        new_ex["combined"] = new_text
        swapped_examples.append(new_ex)

    return Dataset.from_list(swapped_examples)

small_val.set_format(type=None)

# Create swapped validation set
swapped_dataset = swap_evidence(small_val)

# Re-tokenize swapped dataset
swapped_dataset = swapped_dataset.map(tokenize, batched=True)

swapped_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "labels"]
)

# Evaluate
results_swapped = trainer.evaluate(swapped_dataset)

print("Swapped Accuracy:", results_swapped)



Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Swapped Accuracy: {'eval_loss': 1.5622931718826294, 'eval_accuracy': 0.472, 'eval_runtime': 6.5659, 'eval_samples_per_second': 152.303, 'eval_steps_per_second': 9.595, 'epoch': 2.0}


In [18]:
# -----------------------------
# CLAIM-ONLY BASELINE
# -----------------------------

small_train.set_format(type=None)
small_val.set_format(type=None)

def preprocess_claim_only(example):
    example["claim_only"] = example["hypothesis"]
    return example

claim_train = small_train.map(preprocess_claim_only)
claim_val = small_val.map(preprocess_claim_only)

def tokenize_claim(example):
    return tokenizer(
        example["claim_only"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

claim_train = claim_train.map(tokenize_claim, batched=True)
claim_val = claim_val.map(tokenize_claim, batched=True)

# IMPORTANT: labels already exist — do NOT rename

claim_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
claim_val.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

claim_model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)

claim_trainer = Trainer(
    model=claim_model,
    args=training_args,
    train_dataset=claim_train,
    eval_dataset=claim_val,
    compute_metrics=compute_metrics
)

claim_trainer.train()

claim_results = claim_trainer.evaluate()

print("Claim-only Accuracy:", claim_results)


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.871941,0.604000
2,0.915685,0.852806,0.606000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Claim-only Accuracy: {'eval_loss': 0.8528056740760803, 'eval_accuracy': 0.606, 'eval_runtime': 6.5377, 'eval_samples_per_second': 152.959, 'eval_steps_per_second': 9.636, 'epoch': 2.0}


In [19]:
from sklearn.metrics import confusion_matrix
import numpy as np

def get_predictions(trainer, dataset):
    preds = trainer.predict(dataset)
    predictions = np.argmax(preds.predictions, axis=-1)
    labels = preds.label_ids
    return predictions, labels

# Normal predictions
normal_preds, normal_labels = get_predictions(trainer, small_val)

# Swapped predictions
swapped_preds, swapped_labels = get_predictions(trainer, swapped_dataset)

print("Confusion Matrix - Normal")
print(confusion_matrix(normal_labels, normal_preds))

print("\nConfusion Matrix - Swapped")
print(confusion_matrix(swapped_labels, swapped_preds))


Confusion Matrix - Normal
[[311  46  17]
 [ 44 202  48]
 [ 26  55 251]]

Confusion Matrix - Swapped
[[ 46  69 259]
 [  1 119 174]
 [  5  20 307]]


In [20]:
def per_class_accuracy(preds, labels):
    classes = np.unique(labels)
    for c in classes:
        idx = labels == c
        acc = (preds[idx] == labels[idx]).mean()
        print(f"Class {c} accuracy: {acc:.4f}")

print("Normal Per-Class Accuracy")
per_class_accuracy(normal_preds, normal_labels)

print("\nSwapped Per-Class Accuracy")
per_class_accuracy(swapped_preds, swapped_labels)


Normal Per-Class Accuracy
Class 0 accuracy: 0.8316
Class 1 accuracy: 0.6871
Class 2 accuracy: 0.7560

Swapped Per-Class Accuracy
Class 0 accuracy: 0.1230
Class 1 accuracy: 0.4048
Class 2 accuracy: 0.9247


In [21]:
# Claim-only predictions
claim_preds, claim_labels = get_predictions(claim_trainer, claim_val)

print("Confusion Matrix - Claim Only")
print(confusion_matrix(claim_labels, claim_preds))

print("\nClaim-only Per-Class Accuracy")
per_class_accuracy(claim_preds, claim_labels)

Confusion Matrix - Claim Only
[[226  63  85]
 [ 46 186  62]
 [ 62  76 194]]

Claim-only Per-Class Accuracy
Class 0 accuracy: 0.6043
Class 1 accuracy: 0.6327
Class 2 accuracy: 0.5843
